In [ ]:
'''
BT 3 - Model training and classification.ipynb
Author: Muhammmad Talha & Jingchuan Shi
Supervisor: Dr. Ahmed Qureshi
Created 2019/9/6, last modified 2026/02/02 at University of Alberta.
All Rights Reserved.
'''

# Load relevant modules.
import time, json, numpy as np
from pathlib import Path
from nltk.corpus import wordnet as wn
# import spacy
from pathlib import Path


# nlp = spacy.load("en_core_web_lg")          # ← replaces "en_vectors"
try:
    import spacy
    nlp = spacy.load("en_core_web_lg")
except Exception:
    nlp = None


In [3]:
import os, sys
os.environ["TRANSFORMERS_NO_TF"] = "1"   # hard-disable TF path in Transformers
os.environ["USE_TF"] = "0"               # older Transformers also honor this
os.environ["USE_TORCH"] = "1"            # be explicit

# purge anything that might have been imported by mistake
for m in list(sys.modules):
    if m.startswith(("tensorflow", "keras", "tf_keras", "transformers", "sentence_transformers")):
        del sys.modules[m]


In [4]:
from sentence_transformers import SentenceTransformer
st = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
emb = st.encode(["diagnose"], normalize_embeddings=True)
print("dim:", len(emb[0]))  # expect 768 for MPNet


C:\Users\talha4\AppData\Local\anaconda3\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


dim: 768


In [5]:
from pathlib import Path
import json

def set_active_params(thr: float, K: int, cutoff: int, save: bool = True, filename: str = "params.json"):
    thr = float(thr); K = int(K); cutoff = int(cutoff)
    globals().update({
        "THRESH": thr, "K": K, "CUTOFF": cutoff,      # new names
        "THRESHOLD": thr, "KNN_K": K                   # legacy names
    })
    if save:
        Path("results").mkdir(parents=True, exist_ok=True)
        (Path("results")/filename).write_text(json.dumps(
            {"threshold": thr, "K": K, "cutoff": cutoff}, indent=2))

def save_params(params: dict, filename: str):
    Path("results").mkdir(parents=True, exist_ok=True)
    (Path("results")/filename).write_text(json.dumps(params, indent=2))

def _read_params_file(filename: str):
    p = {}
    fp = Path("results")/filename
    if fp.exists():
        try:
            p = json.loads(fp.read_text())
        except Exception:
            pass
    return float(p.get("threshold", 0.18236)), int(p.get("K", 13)), int(p.get("cutoff", 16))

def _get_final_params(prefer: str = "auto"):
    # prefer ∈ {"auto","f1","cost","default"}
    if prefer == "f1":
        return _read_params_file("params_f1.json")
    if prefer == "cost":
        return _read_params_file("params_cost.json")
    if prefer == "default":
        return (0.18236, 13, 16)
    # auto: prefer f1 if present else cost else default
    thr,K,cut = _read_params_file("params_f1.json")
    if (thr, K, cut) != (0.18236, 13, 16):  # crude presence check
        return thr, K, cut
    thr,K,cut = _read_params_file("params_cost.json")
    return (thr, K, cut)

def use_params(kind: str):
    thr, K, cut = _get_final_params(kind)
    set_active_params(thr, K, cut, save=False)
    print(f"Active params ← {kind}: threshold={thr:.3f}, K={K}, cutoff={cut}")


In [6]:
# List of core verbs and their corresponding weights.
knowledge_words = ['list', 'name', 'define', 'repeat', 'state', 'label', 'recall', 'identify', 'reproduce', 'describe', 'recognize', 'select', 'record', 'match', 'relate', 'memorize', 'outline', 'quote', 'enumerate', 'write', 'tell', 'recite', 'cite', 'duplicate', 'read', 'order', 'tabulate', 'draw', 'review', 'indicate', 'underline', 'arrange', 'know', 'point', 'count', 'collect', 'meet', 'study', 'trace', 'find', 'index', 'locate', 'show', 'visualize', 'examine', 'copy', 'sequence', 'acquire', 'retell', 'view', 'observe', 'tally', 'imitate', 'follow']
knowledge_weights = [20, 18, 16, 15, 15, 14, 14, 13, 12, 12, 11, 10, 10, 9, 9, 9, 6, 6, 6, 6, 5, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
comprehension_words = ['explain', 'describe', 'discuss', 'paraphrase', 'restate', 'summarize', 'translate', 'convert', 'review', 'express', 'estimate', 'identify', 'generalize', 'interpret', 'locate', 'give', 'distinguish', 'extend', 'predict', 'recognize', 'defend', 'classify', 'infer', 'report', 'illustrate', 'rewrite', 'select', 'contrast', 'differentiate', 'compare', 'indicate', 'exemplify', 'observe', 'elaborate', 'associate', 'visualize', 'articulate', 'clarify', 'subtract', 'approximate', 'interpolate', 'tell', 'detail', 'outline', 'cite', 'picture', 'interact', 'conclude', 'characterize', 'add', 'factor', 'compute', 'match', 'schedule', 'order', 'sketch', 'draw', 'define', 'operate', 'arrange', 'group', 'extrapolate', 'diagram', 'interrelate', 'represent', 'trace', 'shop', 'suggest', 'understand']
comprehension_weights = [19, 18, 18, 14, 13, 13, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 9, 9, 9, 8, 8, 7, 7, 7, 6, 5, 5, 5, 5, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
application_words = ['demonstrate', 'use', 'apply', 'solve', 'illustrate', 'dramatize', 'practise', 'employ', 'operate', 'sketch', 'prepare', 'show', 'compute', 'relate', 'construct', 'interpret', 'discover', 'change', 'produce', 'manipulate', 'schedule', 'modify', 'predict', 'complete', 'choose', 'classify', 'translate', 'determine', 'examine', 'calculate', 'investigate', 'draw', 'write', 'protect', 'derive', 'chart', 'alphabetize', 'simulate', 'process', 'provide', 'capture', 'project', 'transcribe', 'organize', 'shop', 'establish', 'attain', 'graph', 'assign', 'allocate', 'convert', 'experiment', 'exercise', 'diminish', 'make', 'develop', 'ascertain', 'tabulate', 'depreciate', 'subscribe', 'implement', 'handle', 'transfer', 'factor', 'avoid', 'expose', 'express', 'perform', 'sequence', 'acquire', 'administer', 'personalize', 'adapt', 'plot', 'customize', 'interview', 'paint', 'explore', 'utilize', 'report', 'figure', 'price', 'coordinate', 'simplify', 'consult', 'maintain', 'deliver', 'extend', 'imitate', 'guide', 'conduct', 'multiply', 'build', 'code', 'contribute', 'obtain', 'model', 'compare', 'divide', 'exhibit', 'tally', 'inform', 'diagram', 'expand', 'amend', 'engineer', 'control', 'assess', 'concatenate', 'execute', 'convey', 'articulate', 'restructure', 'criticize', 'appraise', 'participate', 'generalize', 'instruct', 'follow', 'act', 'screen', 'debate', 'question', 'select', 'include', 'dissect', 'retrieve', 'inspect', 'prove', 'inventory', 'respond', 'comply', 'collect']
application_weights = [18, 17, 17, 17, 15, 13, 13, 12, 12, 11, 11, 11, 10, 10, 10, 10, 9, 9, 9, 8, 8, 8, 8, 6, 6, 6, 5, 5, 5, 5, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
analysis_words = ['compare', 'contrast', 'distinguish', 'analyze', 'differentiate', 'separate', 'examine', 'diagram', 'infer', 'categorize', 'experiment', 'discriminate', 'select', 'appraise', 'relate', 'test', 'question', 'classify', 'identify', 'outline', 'illustrate', 'subdivide', 'investigate', 'debate', 'criticize', 'calculate', 'inventory', 'prioritize', 'correlate', 'explain', 'inspect', 'detect', 'dissect', 'manage', 'audit', 'characterize', 'order', 'deduce', 'limit', 'connect', 'diagnose', 'document', 'proofread', 'discover', 'ensure', 'optimize', 'maximize', 'confirm', 'divide', 'transform', 'figure', 'prepare', 'file', 'determine', 'train', 'solve', 'survey', 'group', 'minimize', 'interrupt', 'explore', 'blueprint', 'arrange', 'query', 'edit', 'prove', 'isolate', 'reconcile', 'troubleshoot', 'sketch', 'create', 'summarize', 'dramatize', 'employ', 'inquire', 'link', 'abstract', 'establish', 'organize', 'compute', 'devise', 'moderate', 'delegate', 'research', 'model', 'practise', 'operate', 'demonstrate', 'schedule', 'check', 'use', 'chunk', 'choose', 'scrutinize', 'chart', 'apply', 'allow', 'extrapolate', 'recognize', 'show', 'modify', 'administer', 'review', 'change', 'monitor', 'direct', 'corroborate', 'produce', 'negotiate', 'probe', 'accept', 'design', 'interpret', 'extract', 'manipulate', 'focus', 'write', 'predict', 'resolve']
analysis_weights = [20, 19, 17, 17, 13, 12, 12, 10, 10, 9, 9, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 4, 4, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
synthesis_words = ['design', 'create', 'formulate', 'plan', 'compose', 'construct', 'develop', 'combine', 'assemble', 'propose', 'devise', 'arrange', 'organize', 'collect', 'rearrange', 'prepare', 'reconstruct', 'invent', 'generate', 'modify', 'write', 'categorize', 'rewrite', 'relate', 'compile', 'revise', 'reorganize', 'summarize', 'manage', 'generalize', 'integrate', 'explain', 'produce', 'originate', 'tell', 'incorporate', 'facilitate', 'hypothesize', 'substitute', 'specify', 'improve', 'format', 'correspond', 'model', 'depict', 'synthesize', 'refer', 'comply', 'enhance', 'import', 'overhaul', 'animate', 'predict', 'adapt', 'cultivate', 'code', 'join', 'handle', 'anticipate', 'portray', 'express', 'budget', 'cope', 'debug', 'perform', 'communicate', 'outline', 'prescribe', 'initiate', 'network', 'program', 'lecture', 'dictate', 'advise', 'document', 'gather', 'derive', 'abstract', 'expand', 'establish', 'collaborate', 'conduct', 'contribute', 'coordinate', 'compare', 'speculate', 'simulate', 'progress', 'forecast', 'instruct', 'structure', 'intervene', 'frame', 'measure', 'estimate', 'recommend', 'negotiate', 'consolidate', 'choose', 'contrast', 'imagine', 'individualize', 'recognize', 'solve', 'roleplay', 'review', 'arbitrate', 'teach', 'supervise', 'assess', 'counsel', 'exchange', 'brief', 'reinforce', 'unify', 'pretend', 'update', 'validate']
synthesis_weights = [20, 19, 18, 17, 16, 16, 13, 12, 12, 11, 10, 10, 10, 10, 9, 9, 9, 9, 8, 8, 8, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
evaluation_words = ['judge', 'appraise', 'evaluate', 'support', 'assess', 'select', 'justify', 'compare', 'rate', 'conclude', 'value', 'defend', 'estimate', 'choose', 'critique', 'argue', 'measure', 'recommend', 'discriminate', 'decide', 'interpret', 'criticize', 'contrast', 'rank', 'predict', 'explain', 'summarize', 'score', 'grade', 'revise', 'relate', 'verify', 'test', 'validate', 'attach', 'determine', 'describe', 'convince', 'prescribe', 'consider', 'release', 'counsel', 'hire', 'prioritize', 'deduce', 'enforce', 'advise', 'motivate', 'core', 'uphold', 'resolve', 'reconcile', 'discuss', 'authenticate', 'review', 'monitor', 'weigh', 'debate', 'diagnose', 'infer', 'mediate', 'prove', 'use', 'preserve', 'access', 'consolidate']
evaluation_weights = [21, 17, 17, 15, 15, 14, 14, 13, 13, 12, 10, 10, 10, 9, 9, 9, 9, 9, 8, 7, 7, 7, 6, 6, 6, 6, 6, 5, 5, 4, 4, 4, 4, 4, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
knowledge_words_spacy = nlp(r'list name define repeat state label recall identify reproduce describe recognize select record match relate memorize outline quote enumerate write tell recite cite duplicate read order tabulate draw review indicate underline arrange know point count collect meet study trace find index locate show visualize examine copy sequence acquire retell view observe tally imitate follow')
comprehension_words_spacy = nlp(r'explain describe discuss paraphrase restate summarize translate convert review express estimate identify generalize interpret locate give distinguish extend predict recognize defend classify infer report illustrate rewrite select contrast differentiate compare indicate exemplify observe elaborate associate visualize articulate clarify subtract approximate interpolate tell detail outline cite picture interact conclude characterize add factor compute match schedule order sketch draw define operate arrange group extrapolate diagram interrelate represent trace shop suggest understand')
application_words_spacy = nlp(r'demonstrate use apply solve illustrate dramatize practise employ operate sketch prepare show compute relate construct interpret discover change produce manipulate schedule modify predict complete choose classify translate determine examine calculate investigate draw write protect derive chart alphabetize simulate process provide capture project transcribe organize shop establish attain graph assign allocate convert experiment exercise diminish make develop ascertain tabulate depreciate subscribe implement handle transfer factor avoid expose express perform sequence acquire administer personalize adapt plot customize interview paint explore utilize report figure price coordinate simplify consult maintain deliver extend imitate guide conduct multiply build code contribute obtain model compare divide exhibit tally inform diagram expand amend engineer control assess concatenate execute convey articulate restructure criticize appraise participate generalize instruct follow act screen debate question select include dissect retrieve inspect prove inventory respond comply collect')
analysis_words_spacy = nlp(r'compare contrast distinguish analyze differentiate separate examine diagram infer categorize experiment discriminate select appraise relate test question classify identify outline illustrate subdivide investigate debate criticize calculate inventory prioritize correlate explain inspect detect dissect manage audit characterize order deduce limit connect diagnose document proofread discover ensure optimize maximize confirm divide transform figure prepare file determine train solve survey group minimize interrupt explore blueprint arrange query edit prove isolate reconcile troubleshoot sketch create summarize dramatize employ inquire link abstract establish organize compute devise moderate delegate research model practise operate demonstrate schedule check use chunk choose scrutinize chart apply allow extrapolate recognize show modify administer review change monitor direct corroborate produce negotiate probe accept design interpret extract manipulate focus write predict resolve')
synthesis_words_spacy = nlp(r'design create formulate plan compose construct develop combine assemble propose devise arrange organize collect rearrange prepare reconstruct invent generate modify write categorize rewrite relate compile revise reorganize summarize manage generalize integrate explain produce originate tell incorporate facilitate hypothesize substitute specify improve format correspond model depict synthesize refer comply enhance import overhaul animate predict adapt cultivate code join handle anticipate portray express budget cope debug perform communicate outline prescribe initiate network program lecture dictate advise document gather derive abstract expand establish collaborate conduct contribute coordinate compare speculate simulate progress forecast instruct structure intervene frame measure estimate recommend negotiate consolidate choose contrast imagine individualize recognize solve roleplay review arbitrate teach supervise assess counsel exchange brief reinforce unify pretend update validate')
evaluation_words_spacy = nlp(r'judge appraise evaluate support assess select justify compare rate conclude value defend estimate choose critique argue measure recommend discriminate decide interpret criticize contrast rank predict explain summarize score grade revise relate verify test validate attach determine describe convince prescribe consider release counsel hire prioritize deduce enforce advise motivate core uphold resolve reconcile discuss authenticate review monitor weigh debate diagnose infer mediate prove use preserve access consolidate')
wordlists = [knowledge_words, comprehension_words, application_words, analysis_words, synthesis_words, evaluation_words]
wordlists_spacy = [knowledge_words_spacy, comprehension_words_spacy, application_words_spacy, analysis_words_spacy, synthesis_words_spacy, evaluation_words_spacy]
weights = [knowledge_weights, comprehension_weights, application_weights, analysis_weights, synthesis_weights, evaluation_weights]
namelist = ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']

# Align with BT-2's structure
BASE_DIR    = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
LOGS_DIR    = BASE_DIR / "logs"
MODELS_DIR  = BASE_DIR / "models"
for d in (RESULTS_DIR, LOGS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Canonical files (match BT-2)
CORE_LIST = RESULTS_DIR / "BTverblist_core.txt"
NEW_LIST  = RESULTS_DIR / "BTverblist_new.txt"
EXT_CSV   = RESULTS_DIR / "extended_verbs.csv"
CLF_PKL   = MODELS_DIR  / "bloom_classifier.pkl"

####
core_source_path = CORE_LIST
new_source_path  = NEW_LIST
####
# Read verb lists (UTF-8) if they exist
core_verbs = []
verbs      = []
if CORE_LIST.exists():
    core_verbs = [l.strip() for l in CORE_LIST.read_text(encoding="utf-8").splitlines() if l.strip()]
if NEW_LIST.exists():
    verbs = [l.strip() for l in NEW_LIST.read_text(encoding="utf-8").splitlines() if l.strip()]

In [ ]:
# === Deps, embeddings (cached), and small utils ===
import os, math, glob, json, time, warnings
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd

# ── Install/import sentence-transformers model once (fast at reuse)
try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError("Please install: pip install sentence-transformers") from e

EMBED_CACHE_DIR = os.path.join("results", "_embed_cache")
os.makedirs(EMBED_CACHE_DIR, exist_ok=True)

# Deterministic, normalized embeddings
_st = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def _embed_once(text: str) -> np.ndarray:
    v = _st.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0]
    return v.astype(np.float32, copy=False)

def embed_vec(word: str) -> np.ndarray:
    key = (word or "").strip().lower()
    if not key: key = "empty"
    fp = os.path.join(EMBED_CACHE_DIR, f"{key.replace(' ','_')}.npy")
    if os.path.exists(fp): return np.load(fp)
    v = _embed_once(key); np.save(fp, v); return v

def embed_batch(words: List[str]) -> np.ndarray:
    paths, miss_idx, miss_words, vecs = [], [], [], []
    for i, w in enumerate(words):
        key = (w or "").strip().lower() or "empty"
        fp = os.path.join(EMBED_CACHE_DIR, f"{key.replace(' ','_')}.npy")
        paths.append(fp)
        if os.path.exists(fp): vecs.append(np.load(fp))
        else: vecs.append(None); miss_idx.append(i); miss_words.append(key)
    if miss_words:
        V = _st.encode(miss_words, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
        for j, i in enumerate(miss_idx):
            vecs[i] = V[j]; np.save(paths[i], V[j])
    return np.stack(vecs, axis=0)

LEVELS = ["Kn","Cm","Ap","An","Sn","Ev"]
LVL2IDX = {lv:i for i,lv in enumerate(LEVELS)}
LEVELS_LOWER = ["knowledge","comprehension","application","analysis","synthesis","evaluation"]

def coerce_bloom_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if not all(c in df.columns for c in LEVELS):
        # allow lower-case names
        alt = dict(zip(LEVELS_LOWER, LEVELS))
        for lo, hi in alt.items():
            if lo in df.columns and hi not in df.columns:
                df[hi] = df[lo]
    if not all(c in df.columns for c in LEVELS):
        raise RuntimeError("Need 6 Bloom columns (Kn..Ev) or lowercased equivalents.")
    return df

def find_verb_col(df: pd.DataFrame) -> str:
    for c in ["verb","Verb","word","token","WORD","VERB"]:
        if c in df.columns: return c
    raise RuntimeError("No verb/word/token column found.")

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


In [ ]:
# === Load core lists (+weights) and extended_df ===
def load_core_from_memory_or_results() -> Tuple[List[str], np.ndarray, Optional[Dict[str,int]]]:
    """
    Returns:
      core_words (unique),
      Y_core (multi-hot 6),
      freq (optional dict of weights; None if not available)
    """
    g = globals()
    have_lists = all(n in g for n in [
        "knowledge_words","comprehension_words","application_words","analysis_words","synthesis_words","evaluation_words"
    ])
    have_weights = all(n in g for n in [
        "knowledge_weights","comprehension_weights","application_weights","analysis_weights","synthesis_weights","evaluation_weights"
    ])
    if have_lists:
        core_sets = [
            [w.strip().lower() for w in g["knowledge_words"]],
            [w.strip().lower() for w in g["comprehension_words"]],
            [w.strip().lower() for w in g["application_words"]],
            [w.strip().lower() for w in g["analysis_words"]],
            [w.strip().lower() for w in g["synthesis_words"]],
            [w.strip().lower() for w in g["evaluation_words"]],
        ]
        core_words = sorted(set(w for L in core_sets for w in L))
        word2idx = {w:i for i,w in enumerate(core_words)}
        Y = np.zeros((len(core_words), 6), dtype=int)
        for j, L in enumerate(core_sets):
            for w in L: Y[word2idx[w], j] = 1

        freq = None
        if have_weights:
            weights_sets = [
                g["knowledge_weights"], g["comprehension_weights"], g["application_weights"],
                g["analysis_weights"],  g["synthesis_weights"],  g["evaluation_weights"]
            ]
            # Build max weight per word across levels that include it
            freq = {}
            for j, L in enumerate(core_sets):
                for w, wt in zip(L, weights_sets[j]):
                    freq[w] = max(freq.get(w, 0), int(wt))
        return core_words, Y, freq

    # Fallback: try a core CSV/JSON in results/
    cands = glob.glob(os.path.join("results","*core*.csv")) + glob.glob(os.path.join("results","*core*.json"))
    if not cands:
        raise RuntimeError("Core lists not found (variables or results/*core* file).")
    path = sorted(cands, key=os.path.getmtime, reverse=True)[0]
    df = pd.read_json(path) if path.endswith(".json") else pd.read_csv(path)
    df = coerce_bloom_columns(df)
    vcol = find_verb_col(df)
    df["verb_norm"] = df[vcol].astype(str).str.lower().str.strip()
    agg = df.groupby("verb_norm")[LEVELS].max().reset_index()
    core_words = agg["verb_norm"].tolist()
    Y = agg[LEVELS].astype(int).to_numpy()
    # Optional weights: look for a 'weight' or per-level freq column
    freq = None
    for c in ["weight","freq","frequency"]:
        if c in df.columns:
            freq = df.groupby("verb_norm")[c].max().to_dict()
            break
    return core_words, Y, freq

def load_extended_df_from_memory_or_latest() -> Tuple[List[str], pd.DataFrame]:
    # Prefer extended_df in memory
    if "extended_df" in globals() and isinstance(extended_df, pd.DataFrame):
        df = coerce_bloom_columns(extended_df.copy())
        vcol = find_verb_col(df)
        verbs = df[vcol].astype(str).str.lower().str.strip().tolist()
        return verbs, df

    # Else: newest CSV in results/
    cands = sorted(glob.glob(os.path.join("results","*.csv")), key=os.path.getmtime, reverse=True)
    for p in cands:
        try:
            df = pd.read_csv(p)
            df = coerce_bloom_columns(df)
            vcol = find_verb_col(df)
            verbs = df[vcol].astype(str).str.lower().str.strip().tolist()
            return verbs, df
        except Exception:
            continue
    raise RuntimeError("extended_df not found in memory and no valid CSV in results/.")

core_words, Y_core, core_freq = load_core_from_memory_or_results()
eval_verbs, df_eval = load_extended_df_from_memory_or_latest()

X_core = embed_batch(core_words)
X_eval = embed_batch(eval_verbs)

print(f"[DATA] core={len(core_words)} | eval={len(eval_verbs)} | embed_dim={X_core.shape[1]}")


[DATA] core=358 | eval=30 | embed_dim=768


In [9]:
import numpy as np
import pandas as pd

# Assuming these already exist from your earlier cell:
# core_words, Y_core, core_freq = load_core_from_memory_or_results()
LEVELS = ["Kn", "Cm", "Ap", "An", "Sn", "Ev"]

# --- 1) Summary counts ---
core_n_verbs = len(core_words)
core_entries = int(Y_core.sum())
per_level = dict(zip(LEVELS, Y_core.sum(axis=0).astype(int)))

print(f"# core verbs: {core_n_verbs}")
print(f"# core entries (verb–level assignments): {core_entries}")
print(f"Entry rate (core): {core_entries / core_n_verbs:.3f}")
print("Per level:")
for lev in LEVELS:
    print(f"  {lev}: {per_level[lev]}")

# --- 2) table with one row per (verb, level) assignment ---
rows = []
for i, verb in enumerate(core_words):
    for j, lev in enumerate(LEVELS):
        if Y_core[i, j]:
            rows.append({"verb": verb, "level": lev})

core_long = pd.DataFrame(rows)
core_long.head()


# core verbs: 358
# core entries (verb–level assignments): 559
Entry rate (core): 1.561
Per level:
  Kn: 54
  Cm: 69
  Ap: 133
  An: 119
  Sn: 118
  Ev: 66


,verb,level
0,abstract,An
1,abstract,Sn
2,accept,An
3,access,Ev
4,acquire,Kn


In [ ]:
# === structural metrics + SCS (multi-label, gap-aware, peer-backed) ===
import math, numpy as np
from typing import List, Optional, Tuple
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ------------------------------------------------------------
# Acceptance / entry bookkeeping (BT4 style; for reporting)
# ------------------------------------------------------------
def acceptance_entry(Y_bin: np.ndarray) -> dict:
    """
    Returns:
      acceptance = (# items with ≥1 label) / N
      entry_rate = average #labels per item
      en_over_ac = entry_rate / acceptance  (≈ avg #domains per accepted item)
      per_level  = counts per level (length 6)
    """
    n = Y_bin.shape[0]
    lbls = Y_bin.sum(axis=1)
    acc = float((lbls > 0).sum()) / max(n, 1)
    en  = float(lbls.mean()) if n else 0.0
    ratio = (en/acc) if acc > 0 else 0.0
    per_level = Y_bin.sum(axis=0).astype(int).tolist()
    return dict(acceptance=acc, entry_rate=en, en_over_ac=ratio, per_level=per_level)

# ------------------------------------------------------------
# Centroids, gap distances, and assignments (multi-label)
# ------------------------------------------------------------
def _l2_normalize(M: np.ndarray, axis: int = 1, eps: float = 1e-12) -> np.ndarray:
    nrm = np.linalg.norm(M, axis=axis, keepdims=True)
    nrm = np.where(nrm < eps, 1.0, nrm)
    return M / nrm

# --- helper: safe Spearman that returns 0.0 for constant/invalid inputs ---
def _safe_spearman(xs, ys):
    xs = np.asarray(xs, float); ys = np.asarray(ys, float)
    if ys.size == 0 or not np.all(np.isfinite(ys)):
        return 0.0
    # constant series -> undefined rho; treat as no trend (0.0)
    if len(set(np.round(ys, 12))) <= 1:
        return 0.0
    try:
        rho, _ = spearmanr(xs, ys)
        return float(rho) if np.isfinite(rho) else 0.0
    except Exception:
        return 0.0

def vector_centroid_summary(
    Y_bin: np.ndarray,
    words: List[str],
    V_all: Optional[np.ndarray] = None
) -> Tuple[np.ndarray, np.ndarray, dict, float, list, list]:
    """
    Build per-level centroids and summarize gap-wise distances & monotonicity.

    Equations (cosine geometry; all vectors L2-normalized):

      Level centroid:
        μ_ℓ = mean_{x ∈ L_ℓ} x ;  then μ_ℓ ← μ_ℓ / ||μ_ℓ||

      Between-centroid distance for gap k (k=1..5):
        D_k = mean_{i=0..5-k} [ 1 - cos( μ_i , μ_{i+k} ) ]

      Monotonic trend:
        ρ = Spearman( k , D_k ),  k∈{1..5}

    Returns:
      C        : (6,d) centroids (unit vectors; 0 if empty level)
      Dmat     : (6,6) symmetric cosine-distance matrix between centroids
      D_summ   : {k: (mean, max, min)} for k=1..5
      rho      : Spearman rho over (k, D_k)
      counts   : list of |L_ℓ| per level
      assign   : list[6] of np.array indices for accepted items per level
    """
    n_items = Y_bin.shape[0]
    # Embeddings for ALL words (compute once per call)
    if V_all is None:
        V_all = embed_batch(words).astype(np.float64)
    V_all = _l2_normalize(V_all, axis=1)

    # Indices of accepted items per level (multi-label allowed)
    assign = [np.where(Y_bin[:, j] == 1)[0] for j in range(6)]
    counts = [int(len(idx)) for idx in assign]

    # Centroids per level
    d = V_all.shape[1] if n_items > 0 else len(embed_vec("probe"))
    C = np.zeros((6, d), dtype=np.float64)
    for j in range(6):
        idx = assign[j]
        if len(idx) == 0: 
            continue
        c = V_all[idx].mean(axis=0)
        C[j] = _l2_normalize(c.reshape(1,-1), axis=1)[0]

    # Cosine distance matrix between centroids
    Dmat = np.zeros((6, 6), dtype=float)
    for i in range(6):
        for j in range(i+1, 6):
            sim = float(np.dot(C[i], C[j]))
            dist = 1.0 - sim  # in [0,2]
            Dmat[i, j] = Dmat[j, i] = max(0.0, min(2.0, dist))

    # Summaries by gap k and monotonic trend
    D_summ, means_by_k = {}, []
    for k in range(1, 6):
        vals = [Dmat[i, i+k] for i in range(0, 6-k)]
        m = float(np.mean(vals)) if len(vals) else float("nan")
        D_summ[k] = (m, float(np.max(vals)) if len(vals) else float("nan"),
                        float(np.min(vals)) if len(vals) else float("nan"))
        means_by_k.append(0.0 if math.isnan(m) else m)
    ks = np.arange(1, 6, dtype=float)
    rho = _safe_spearman(ks, means_by_k)
    return C, Dmat, D_summ, float(rho), counts, assign


# ------------------------------------------------------------
# Mutual influence (adjacent Jaccard overlap)
# ------------------------------------------------------------
def mutual_influence(Y_bin: np.ndarray):
    """
    Jaccard overlap between accepted sets:
      J(i,j) = |S_i ∩ S_j| / |S_i ∪ S_j|,  bounded in [0,1]
    We summarize gap-wise means:
      M_k = mean_{i} J(i, i+k)
    """
    sets = [set(np.where(Y_bin[:, j] == 1)[0].tolist()) for j in range(6)]
    J = np.zeros((6,6), dtype=float)
    for i in range(6):
        for j in range(6):
            if i == j:
                J[i,j] = 1.0
            else:
                u = len(sets[i] | sets[j]); inter = len(sets[i] & sets[j])
                J[i,j] = (inter / u) if u > 0 else 0.0
    summ = {}
    for k in range(1,6):
        vals = [J[i, i+k] for i in range(0, 6-k)]
        summ[k] = (float(np.mean(vals)), float(np.max(vals)), float(np.min(vals))) if vals else (float('nan'),)*3
    return J, summ

# ------------------------------------------------------------
# Per-class peer-reviewed coherence (Silhouette + DB, one-vs-rest)
# ------------------------------------------------------------


def _coherence_per_class(V_all: np.ndarray, Y_bin: np.ndarray, use_cosine: bool = True) -> Tuple[float, dict]:
    """
    For each level ℓ, form binary labels y^{(ℓ)} over ALL items and compute:
      Sil_ℓ  = Silhouette(V_all, y^{(ℓ)}, metric='cosine' if use_cosine else 'euclidean') ∈ [-1,1]
      DB_ℓ   = Davies–Bouldin(V_all, y^{(ℓ)}) ∈ [0,∞), lower better
    Normalize to higher-is-better in [0,1]:
      S̃_ℓ = (Sil_ℓ + 1) / 2
      D̃_ℓ = 1 / (1 + DB_ℓ)
    Aggregate (skip degenerate cases automatically):
      S̄ = mean_ℓ S̃_ℓ,   D̄ = mean_ℓ D̃_ℓ
      C  = 0.7·S̄ + 0.3·D̄
    """
    # Normalize embeddings if using cosine
    V = _l2_normalize(V_all, axis=1) if use_cosine else V_all

    S_norm, D_norm = [], []
    details = {"sil": [], "db": [], "S_tilde": [], "D_tilde": [],
               "valid_idx": [], "skipped": []}

    n = V.shape[0]
    metric = "cosine" if use_cosine else "euclidean"

    for lvl in range(6):
        y = (Y_bin[:, lvl] == 1).astype(int)
        n_pos = int(y.sum()); n_neg = int(n - n_pos)

        # Need at least 2 samples in each cluster for silhouette to be valid
        if n_pos < 2 or n_neg < 2:
            details["skipped"].append(lvl)
            continue

        # Silhouette
        try:
            sil = float(silhouette_score(V, y, metric=metric))
            Sil_t = (sil + 1.0) / 2.0
        except Exception:
            sil, Sil_t = float("nan"), float("nan")

        # Davies–Bouldin
        try:
            db = float(davies_bouldin_score(V, y))
            DB_t = 1.0 / (1.0 + db)
        except Exception:
            db, DB_t = float("nan"), float("nan")

        details["sil"].append(sil)
        details["db"].append(db)
        details["S_tilde"].append(Sil_t)
        details["D_tilde"].append(DB_t)
        details["valid_idx"].append(lvl)

        if not np.isnan(Sil_t): S_norm.append(Sil_t)
        if not np.isnan(DB_t):  D_norm.append(DB_t)

    if len(S_norm) == 0 and len(D_norm) == 0:
        details.update({"S_bar": float("nan"), "D_bar": float("nan"), "C": 0.0})
        return 0.0, details

    S_bar = float(np.mean(S_norm)) if len(S_norm) else 0.0
    D_bar = float(np.mean(D_norm)) if len(D_norm) else 0.0
    C = 0.7 * S_bar + 0.3 * D_bar
    details.update({"S_bar": S_bar, "D_bar": D_bar, "C": C})
    return float(C), details



# ------------------------------------------------------------
# Master evaluator: packages everything needed for SCS★
# ------------------------------------------------------------
def structural_eval(
    Y_bin: np.ndarray,
    words: List[str],
    V_all: Optional[np.ndarray] = None
) -> dict:
    """
    Returns a dict with:
      acc            : acceptance/entry metrics (reporting only)
      centroids      : (6,d) unit centroids
      V_all          : (n,d) unit embeddings for all words
      assign_idx     : list of 6 index arrays (accepted items per level)
      D_summ         : gap-wise centroid distance summary {k: (mean,max,min)}
      trend_rho      : Spearman rho over (k, D_k)
      level_counts   : list of |L_ℓ|
      M_summ         : mutual influence (Jaccard) summary by gap
      C_peer         : per-class peer coherence term (0..1) and details
    """
    acc = acceptance_entry(Y_bin)
    # Precompute embeddings once
    if V_all is None:
        V_all = embed_batch(words).astype(np.float64)
    V_all = _l2_normalize(V_all, axis=1)

    Cents, Dmat, D_summ, rho, counts, assign = vector_centroid_summary(Y_bin, words, V_all=V_all)
    J, J_summ = mutual_influence(Y_bin)
    C_peer, C_details = _coherence_per_class(V_all, Y_bin)

    return dict(
        acc=acc,
        centroids=Cents, V_all=V_all, assign_idx=assign,
        D_summ=D_summ, trend_rho=rho, level_counts=counts,
        M_summ=J_summ,
        C_peer=C_peer, C_details=C_details
    )

# ------------------------------------------------------------
# SCS: multi-label, gap-aware structure with peer coherence
# ------------------------------------------------------------
def SCS(
    out: dict,
    *,
    wD: float = 1.0, wM: float = 1.0, wT: float = 0.8, wC: float = 0.3, wE: float = 0.05,
    empty_min: int = 10,
    target_ratio: float = 1.51, acc_window=(0.30, 0.85)
):
    r"""
    Final score:
        SCS = 1·D  − 1·M  + 0.8·T  + 0.3·C  − 0.05·E

    Components (all in [0,1], except E is an integer):

      1) Gap-aware separation with compactness (multi-label):
         Let D_k be mean centroid cosine distance for gap k:
             D_k = mean_i [ 1 − cos( μ_i , μ_{i+k} ) ]
         Weighted between:
             \bar{D} = (Σ_{k=1..5} k·D_k) / (Σ_{k=1..5} k)
         Within scatter per level ℓ:
             W_ℓ = mean_{x∈L_ℓ} [ 1 − cos( x , μ_ℓ ) ]
             \bar{W} = (Σ_ℓ |L_ℓ|·W_ℓ) / (Σ_ℓ |L_ℓ|)
         Bounded separation:
             D = \bar{D} / ( \bar{D} + \bar{W} + ε )

      2) Adjacent overlap (mutual influence):
             M = mean_ℓ  Jaccard( S_ℓ , S_{ℓ+1} )

      3) Monotonic trend across gaps:
             T = ( Spearman( {k},{D_k} ) + 1 ) / 2

      4) Per-class peer-reviewed coherence (one-vs-rest):
             C = 0.7·mean_ℓ[(Sil_ℓ+1)/2] + 0.3·mean_ℓ[ 1/(1+DB_ℓ) ]

      5) Empty-level guardrail:
             E = # { ℓ : |L_ℓ| < empty_min }
    """
    eps = 1e-6

    # --- Degenerate guard: if no accepted items at all, hard-penalize ---
    counts = out.get("level_counts", [0]*6)
    if sum(int(c) for c in counts) == 0:
        return -1e9, dict(D=0.0, M=1.0, trend=0.0, C=0.0, empty_levels=6,
                          acceptance=0.0, overlap=0.0, D_bar=0.0, W_bar=0.0, rho=0.0)

    # --- D: gap-aware separation with compactness (bounded) ---
    D_means = [out["D_summ"][k][0] for k in range(1, 6)]
    D_means = [0.0 if (m is None or not np.isfinite(m)) else float(m) for m in D_means]
    D_bar = float(np.average(D_means, weights=[1,2,3,4,5])) if np.any(np.isfinite(D_means)) else 0.0

    V = out["V_all"]; Cents = out["centroids"]; assign = out["assign_idx"]
    W_list, n_list = [], []
    for j in range(6):
        idx = assign[j]
        if len(idx) == 0: 
            continue
        xj = V[idx]; cj = Cents[j].reshape(1, -1)
        Wj = float(np.mean(1.0 - np.clip(xj @ cj.T, -1.0, 1.0)))
        if np.isfinite(Wj):
            W_list.append(Wj); n_list.append(len(idx))
    W_bar = float(np.average(W_list, weights=n_list)) if n_list else 0.0

    D_term = D_bar / (D_bar + W_bar + eps)
    D_term = float(np.clip(D_term, 0.0, 1.0))

    # --- M: adjacent overlap (lower better) ---
    M_k1 = out["M_summ"][1][0]
    M_term = float(np.clip(0.0 if (M_k1 is None or not np.isfinite(M_k1)) else M_k1, 0.0, 1.0))

    # --- T: monotonic trend (rho -> [0,1]) ---
    rho = out.get("trend_rho", 0.0)
    if rho is None or not np.isfinite(rho): rho = 0.0
    T_term = float(np.clip((rho + 1.0) / 2.0, 0.0, 1.0))

    # --- C: per-class peer-reviewed coherence (0..1) ---
    C_val = out.get("C_peer", 0.0)
    if C_val is None or not np.isfinite(C_val): C_val = 0.0
    C_term = float(np.clip(C_val, 0.0, 1.0))

    # --- E: #near-empty levels (< empty_min) ---
    E = int(sum(1 for c in counts if c < int(empty_min)))

    score = (wD * D_term) - (wM * M_term) + (wT * T_term) + (wC * C_term) - (wE * E)

    # ---- Presentation-only (NOT part of score) ----
    acc = float(out["acc"].get("acceptance", 0.0) or 0.0)
    en_ac = float(out["acc"].get("en_over_ac", 0.0) or 0.0)

    overlap_term = 0.0
    if acc > 0:
        overlap_term = 1.0 - abs(en_ac - target_ratio) / max(target_ratio, 1e-6)
        overlap_term = float(np.clip(overlap_term, 0.0, 1.0))

    lo, hi = acc_window
    if acc < lo:       acc_term = max(0.0, 1.0 - (lo - acc)/max(lo,1e-6))
    elif acc > hi:     acc_term = max(0.0, 1.0 - (acc - hi)/max(1.0-hi,1e-6))
    else:              acc_term = 1.0

    parts = dict(
        D=D_term, M=M_term, trend=T_term, C=C_term, empty_levels=E,
        acceptance=acc_term, overlap=overlap_term,
        D_bar=D_bar, W_bar=W_bar, rho=rho
    )
    return float(score), parts


In [39]:
# === MODEL ZOO + Taxonomy-aware AutoTune (WEIGHTED + OOF, rank-aware, robust) ===
import numpy as np, pandas as pd, warnings
from itertools import product
from typing import List, Dict, Any, Tuple
import numpy.linalg as npl

from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier, PassiveAggressiveClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA, QuadraticDiscriminantAnalysis as QDA
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import NearestCentroid, KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import KFold

# ------------------------------
# Toggles & constants
# ------------------------------
RNG_SEED = 13
USE_ROW_WEIGHTS = True
USE_OOF = True          # <-- OOF calibration on by default
N_SPLITS_OOF = 5
SCS_EMPTY_LEVEL_PEN = 0.05  # penalty per empty level in SCS objective

# Warnings hygiene
warnings.filterwarnings("ignore", message="Variables are collinear")
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ------------------------------
# Targets (acceptance/en_over_ac bands) + penalties
# ------------------------------
def core_targets_from_labels(Y_core, margin_acc=0.08, margin_enac=0.08, bootstrap=400, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    n = Y_core.shape[0]
    accs, enacs = [], []
    for _ in range(bootstrap):
        idx = rng.integers(0, n, n)
        Yb = Y_core[idx]
        lbls = Yb.sum(axis=1)
        acc  = float((lbls > 0).mean())
        en   = float(lbls.mean())
        enac = (en/acc) if acc > 0 else 0.0
        accs.append(acc); enacs.append(enac)
    acc_med, enac_med = float(np.median(accs)), float(np.median(enacs))
    ACC_BAND  = (max(0.01, acc_med - margin_acc), min(0.99, acc_med + margin_acc))
    ENAC_BAND = (max(1.0,  enac_med - margin_enac), enac_med + margin_enac)
    if ACC_BAND[1] <= ACC_BAND[0]: ACC_BAND = (ACC_BAND[0], ACC_BAND[0] + 1e-3)
    if ENAC_BAND[1] <= ENAC_BAND[0]: ENAC_BAND = (ENAC_BAND[0], ENAC_BAND[0] + 1e-3)
    return dict(ACC_BAND=ACC_BAND, ENAC_BAND=ENAC_BAND, SCS_TIE_EPS=0.02, PEN_ACC=5.0, PEN_ENAC=2.5)

TARGETS = core_targets_from_labels(Y_core)
print("Data-driven TARGETS:", TARGETS)

# Clamp acceptance band for realism & valid order
lo, hi = TARGETS["ACC_BAND"]
lo = max(0.25, min(lo, 0.85))
hi = max(lo + 1e-3, min(0.85, hi))
TARGETS["ACC_BAND"] = (lo, hi)
print("Adjusted TARGETS:", TARGETS)

def _penalty_from_out(out, ACC_BAND, ENAC_BAND, pen_acc=5.0, pen_enac=2.5):
    acc   = float(out["acc"]["acceptance"])
    enac  = float(out["acc"]["en_over_ac"])
    lo, hi = ACC_BAND
    loe, hie = ENAC_BAND
    p_acc  = 0.0 if (lo <= acc <= hi) else min(abs(acc-lo), abs(acc-hi))**2 * pen_acc
    p_enac = 0.0 if (loe <= enac <= hie) else min(abs(enac-loe), abs(enac-hie))**2 * pen_enac
    return p_acc + p_enac

def _objective(scs, out, TARGETS):
    pen = _penalty_from_out(out, TARGETS["ACC_BAND"], TARGETS["ENAC_BAND"],
                            TARGETS["PEN_ACC"], TARGETS["PEN_ENAC"])
    return float(scs) - float(pen)

def _pareto_front(df, cols=[("Eval_SCS", False), ("Eval_Acceptance", False)]):
    X = df.copy()
    for c, asc in cols:
        if asc: X[c] = -X[c]
    keep, vals = [], X[[c for c,_ in cols]].to_numpy()
    for i in range(len(X)):
        v = vals[i]; dominated = False
        for j in range(len(X)):
            if i == j: continue
            w = vals[j]
            if np.all(w >= v) and np.any(w > v):
                dominated = True; break
        if not dominated: keep.append(i)
    return df.iloc[keep].copy()

# ------------------------------
# Row weights
# ------------------------------
def _build_core_row_weights() -> np.ndarray:
    n = X_core.shape[0]
    w = np.ones(n, dtype=float)
    if 'CORE_WEIGHTS_ROW' in globals():
        w = np.asarray(CORE_WEIGHTS_ROW, dtype=float)
        if w.shape[0] != n: raise ValueError("CORE_WEIGHTS_ROW length mismatch.")
    elif 'core_weight_dict' in globals() and 'core_words' in globals():
        w = np.array([float(globals()['core_weight_dict'].get(core_words[i], 1.0)) for i in range(n)], dtype=float)
    elif 'core_freq' in globals() and 'core_words' in globals():
        raw = np.array([float(globals()['core_freq'].get(core_words[i], 0.0)) for i in range(n)], dtype=float)
        w = 1.0 + np.log1p(raw)
    w = np.nan_to_num(w, nan=1.0, posinf=3.0, neginf=0.5)
    w = np.clip(w, 0.5, 3.0)
    m = w.mean()
    if m > 0: w = w / m
    return w

ROW_W = _build_core_row_weights() if USE_ROW_WEIGHTS else np.ones(X_core.shape[0], dtype=float)
print(f"[weights] use={USE_ROW_WEIGHTS} | min={ROW_W.min():.3f} mean={ROW_W.mean():.3f} max={ROW_W.max():.3f}")

# ------------------------------
# Rank helpers (for LDA/QDA)
# ------------------------------
def _effective_rank(A: np.ndarray, eps: float = 1e-10) -> int:
    try:
        s = npl.svd(A, full_matrices=False, compute_uv=False)
        if s.size == 0: return 0
        tol = eps * max(A.shape) * (s[0] if np.isfinite(s[0]) else 1.0)
        return int(np.sum(s > tol))
    except Exception:
        return int(max(0, min(A.shape[0]-1, A.shape[1])))

def _pca_dim_for_lda_qda(X: np.ndarray, n_classes: int = 2, hard_cap: int = 256) -> int:
    r = _effective_rank(X)
    if r <= 2: return 0
    return int(max(2, min(hard_cap, r, n_classes - 1 if n_classes > 1 else r)))

print("effective rank X_core:", _effective_rank(X_core))

# ------------------------------
# Scoring + robust CDF calibration
# ------------------------------
def scores_ovr(clf, X) -> np.ndarray:
    n = X.shape[0]
    if hasattr(clf, "decision_function"):
        try:
            S = np.asarray(clf.decision_function(X), dtype=float)
            return S.reshape(n, -1)
        except Exception:
            pass
    ests = getattr(clf, "estimators_", None)
    if not ests or len(ests) != 6:
        raise RuntimeError("OvR classifier is not fitted or missing .estimators_ (expected 6).")
    S_all = np.zeros((n, 6), dtype=float)
    for j, est in enumerate(ests):
        model = est.steps[-1][1] if hasattr(est, "steps") else est
        if hasattr(model, "decision_function"):
            s = est.decision_function(X); s = np.asarray(s, dtype=float)
            S_all[:, j] = s.ravel() if s.ndim > 1 else s
            continue
        if hasattr(model, "predict_proba"):
            P = np.asarray(est.predict_proba(X))
            classes_ = getattr(model, "classes_", getattr(est, "classes_", np.array([0,1])))
            if P.ndim == 1 or (P.ndim == 2 and P.shape[1] == 1):
                only_cls = int(classes_[0]) if classes_.size else 0
                S_all[:, j] = 1.0 if only_cls == 1 else 0.0
            else:
                try:
                    idx_pos = int(np.where(classes_ == 1)[0][0])
                    S_all[:, j] = P[:, idx_pos]
                except Exception:
                    S_all[:, j] = P.max(axis=1)
            continue
        yhat = est.predict(X)
        S_all[:, j] = np.asarray(yhat, dtype=float)
    return S_all

def _fit_cdf_per_class(S_core: np.ndarray) -> List[Tuple[np.ndarray, np.ndarray]]:
    # robust CDF even for constant/NaN/Inf
    S_core = np.nan_to_num(S_core, nan=0.0, posinf=0.0, neginf=0.0)
    cdfs = []
    for j in range(S_core.shape[1]):
        x = np.asarray(S_core[:, j], dtype=float)
        mask = np.isfinite(x)
        if not mask.any():
            xs = np.array([0.0]); cdf = np.array([1.0]); cdfs.append((xs, cdf)); continue
        xs = np.sort(x[mask])
        if xs.size == 0:
            xs = np.array([0.0]); cdf = np.array([1.0]); cdfs.append((xs, cdf)); continue
        if np.allclose(xs[0], xs[-1]):
            v = xs[0]
            xs = np.array([v - 1e-9, v + 1e-9]); cdf = np.array([0.0, 1.0])
            cdfs.append((xs, cdf)); continue
        ranks = np.arange(1, len(xs)+1, dtype=float)
        cdf = ranks / (len(xs) + 1.0)
        cdfs.append((xs, cdf))
    return cdfs

def _apply_cdf_per_class(S: np.ndarray, cdfs: List[Tuple[np.ndarray, np.ndarray]]) -> np.ndarray:
    S = np.nan_to_num(S, nan=0.0, posinf=0.0, neginf=0.0)
    S01 = np.zeros_like(S, dtype=float)
    for j in range(S.shape[1]):
        xs, cdf = cdfs[j]
        S01[:, j] = np.interp(S[:, j], xs, cdf, left=cdf[0], right=cdf[-1])
    return np.clip(S01, 0.0, 1.0)

# ------------------------------
# Weighted thresholds & evaluation
# ------------------------------
P_GRID = list(range(20, 61, 5))
SCS_TIE_EPS = TARGETS["SCS_TIE_EPS"]

def _weighted_quantile(x: np.ndarray, w: np.ndarray, q: float) -> float:
    x = np.asarray(x, float); w = np.asarray(w, float)
    if x.size == 0: return 1.0
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[m], w[m]
    if x.size == 0: return 1.0
    ix = np.argsort(x); x, w = x[ix], w[ix]
    cw = np.cumsum(w)
    if cw[-1] <= 0: return float(x[-1])
    t = q * cw[-1]
    k = np.searchsorted(cw, t, side="right")
    k = min(max(k, 0), len(x)-1)
    return float(x[k])

def thresholds_from_percentile_weighted(S_core01: np.ndarray, Y_core: np.ndarray, p: int, row_w: np.ndarray) -> np.ndarray:
    thr = np.zeros(6, dtype=float)
    q = p / 100.0
    for j in range(6):
        pos = np.where(Y_core[:, j] == 1)[0]
        if len(pos):
            thr[j] = _weighted_quantile(S_core01[pos, j], row_w[pos], q)
        else:
            thr[j] = _weighted_quantile(S_core01[:, j], row_w, q)
        thr[j] = float(np.clip(thr[j], 0.0, 1.0))
    return thr

def evaluate_with_thresholds(S_core01, S_eval01, thr, verbs_core, verbs_eval):
    Yb_core = (S_core01 >= thr.reshape(1,-1)).astype(int)
    Yb_eval = (S_eval01 >= thr.reshape(1,-1)).astype(int)
    out_c = structural_eval(Yb_core, verbs_core)
    out_e = structural_eval(Yb_eval, verbs_eval)
    scs_c, parts_c = SCS(out_c)
    scs_e, parts_e = SCS(out_e)
    return (Yb_core, Yb_eval, out_c, out_e, scs_c, scs_e, parts_c, parts_e)

# ------------------------------
# Model-specific build (PCA for LDA/QDA) + fit wrappers + OOF
# ------------------------------
def make_estimator(label: str, base, params: dict, X_fit: np.ndarray, y_fit: np.ndarray):
    if label in ("LDA", "QDA"):
        n_comp = _pca_dim_for_lda_qda(X_fit, n_classes=2, hard_cap=256)
        if n_comp < 2:
            raise RuntimeError(f"{label}: insufficient rank ({n_comp}) after analysis.")
        pca = PCA(n_components=n_comp, whiten=True, random_state=RNG_SEED)
        clf = base(**params)
        return Pipeline([("pca", pca), ("clf", clf)])
    return base(**params)

def _fit_ovr_core(label, base, params, X, Y, row_w):
    est = make_estimator(label, base, params, X, Y)
    ovr = OneVsRestClassifier(est)
    try:
        ovr.fit(X, Y, sample_weight=row_w)
    except TypeError:
        ovr.fit(X, Y)
    return ovr

def _scores_in_sample(label, base, params, X_core, Y_core, X_eval, row_w):
    clf = _fit_ovr_core(label, base, params, X_core, Y_core, row_w)
    S_core_raw = scores_ovr(clf, X_core)
    S_eval_raw = scores_ovr(clf, X_eval)
    return S_core_raw, S_eval_raw, clf

def _scores_oof(label, base, params, X, Y, row_w, n_splits=5, seed=RNG_SEED):
    # Robust OOF; if any fold fails, raise to allow fallback
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    S_oof = np.zeros((X.shape[0], Y.shape[1]), dtype=float)
    for tr, va in kf.split(X):
        est = make_estimator(label, base, dict(params), X[tr], Y[tr])
        ovr = OneVsRestClassifier(est)
        try:
            ovr.fit(X[tr], Y[tr], sample_weight=row_w[tr] if row_w is not None else None)
        except TypeError:
            ovr.fit(X[tr], Y[tr])
        S_oof[va] = scores_ovr(ovr, X[va])
    clf_full = _fit_ovr_core(label, base, params, X, Y, row_w)
    return S_oof, clf_full

def _get_scores(label, base, params):
    # choose OOF with fallback to in-sample (prevents "no calibration found" situations)
    row_w = ROW_W if USE_ROW_WEIGHTS else None
    if USE_OOF:
        try:
            S_core_raw, clf = _scores_oof(label, base, params, X_core, Y_core, row_w, n_splits=N_SPLITS_OOF, seed=RNG_SEED)
            S_eval_raw = scores_ovr(clf, X_eval)
            return S_core_raw, S_eval_raw, clf, True
        except Exception as e:
            print(f"[warn] OOF failed for {label} {params}: {e}. Falling back to in-sample calibration.")
    S_core_raw, S_eval_raw, clf = _scores_in_sample(label, base, params, X_core, Y_core, X_eval, row_w)
    return S_core_raw, S_eval_raw, clf, False

# ------------------------------
# Legacy kNN (vote) with guards
# ------------------------------
def _safe_cosine(S):
    return np.clip(S, -1.0, 1.0)

def legacy_knn_predict_eval(X_eval, X_core, Y_core, K, thresh, cutoff, freq=None, core_vocab=None):
    S = _safe_cosine(X_eval @ X_core.T)
    n = X_eval.shape[0]
    Yb = np.zeros((n, 6), dtype=int); Ys = np.zeros((n, 6), dtype=float)
    def _weights(top_idx):
        if freq is None: return np.ones(len(top_idx), float)
        if isinstance(freq, dict) and core_vocab is not None:
            return np.array([freq.get(core_vocab[j], 1.0) for j in top_idx], dtype=float)
        f = np.asarray(freq, dtype=float); 
        return f[top_idx] if f.ndim == 1 else np.ones(len(top_idx), float)
    for i in range(n):
        s = S[i]; idx = np.where(s >= float(thresh))[0]
        if idx.size == 0: continue
        top = idx[np.argsort(s[idx])[::-1][:int(K)]]
        w = _weights(top).reshape(-1,1)
        sc = (Y_core[top] * w).sum(axis=0)
        Ys[i] = sc; Yb[i] = (sc >= float(cutoff)).astype(int)
    return Yb, Ys

def legacy_knn_predict_core_loo(X_core, Y_core, K, thresh, cutoff, freq=None, core_vocab=None):
    S = _safe_cosine(X_core @ X_core.T); np.fill_diagonal(S, 0.0)
    n = X_core.shape[0]; Yb = np.zeros((n, 6), dtype=int)
    def _weights(top_idx):
        if freq is None: return np.ones(len(top_idx), float)
        if isinstance(freq, dict) and core_vocab is not None:
            return np.array([freq.get(core_vocab[j], 1.0) for j in top_idx], dtype=float)
        f = np.asarray(freq, dtype=float); 
        return f[top_idx] if f.ndim == 1 else np.ones(len(top_idx), float)
    for i in range(n):
        s = S[i]; idx = np.where(s >= float(thresh))[0]
        if idx.size == 0: continue
        top = idx[np.argsort(s[idx])[::-1][:int(K)]]
        w = _weights(top).reshape(-1,1)
        sc = (Y_core[top] * w).sum(axis=0)
        Yb[i] = (sc >= float(cutoff)).astype(int)
    return Yb

def tune_legacy_knn(
    Ks=(5,7,9,11,13,15),
    Th=(0.12,0.14,0.16,0.20,0.24,0.28,0.32),
    Ct=(6,8,10,12,16,20,24,28,32),
    use_freq=False,
    min_accept: float = 0.01
):
    freq  = core_freq  if (use_freq and 'core_freq'  in globals()) else None
    vocab = core_words if               'core_words' in globals()  else None
    best = None
    for K, th, co in product(Ks, Th, Ct):
        Yb_eval, _ = legacy_knn_predict_eval(X_eval, X_core, Y_core, K, th, co, freq=freq, core_vocab=vocab)
        out_e = structural_eval(Yb_eval, eval_verbs)
        if out_e["acc"]["acceptance"] < min_accept:
            continue
        scs_e, parts_e = SCS(out_e)
        Yb_core = legacy_knn_predict_core_loo(X_core, Y_core, K, th, co, freq=freq, core_vocab=vocab)
        out_c = structural_eval(Yb_core, core_words); scs_c, parts_c = SCS(out_c)
        obj_e = _objective(scs_e, out_e, TARGETS)
        rec = dict(Model="Legacy kNN", Params=dict(K=K, thresh=th, cutoff=co, use_freq=bool(use_freq)),
                   Mode=f"K={K}/th={th}/c={co}", thr=None,
                   scs_c=scs_c, scs_e=scs_e, obj_e=obj_e,
                   out_c=out_c, out_e=out_e, parts_c=parts_c, parts_e=parts_e,
                   Yb_c=Yb_core, Yb_e=Yb_eval, clf=None, S_core=None, S_eval=None, cdfs=None, row_w=ROW_W)
        if (best is None) or (rec["scs_e"] > best["scs_e"]) or (
            abs(rec["scs_e"] - best["scs_e"]) < SCS_TIE_EPS and
            rec["out_e"]["acc"]["acceptance"] > best["out_e"]["acc"]["acceptance"]
        ):
            best = rec
    return best

# ------------------------------
# Model grids (family-specific)
# ------------------------------
MODEL_GRID = [
    ("LinearSVC", LinearSVC, [
        {"C":0.1, "class_weight":"balanced", "max_iter":5000, "tol":1e-4, "random_state":RNG_SEED},
        {"C":0.5, "class_weight":"balanced", "max_iter":5000, "tol":1e-4, "random_state":RNG_SEED},
        {"C":1.0, "class_weight":"balanced", "max_iter":5000, "tol":1e-4, "random_state":RNG_SEED},
        {"C":2.0, "class_weight":"balanced", "max_iter":5000, "tol":1e-4, "random_state":RNG_SEED},
    ]),
    ("CalibratedSVC", lambda **kw: CalibratedClassifierCV(
        base_estimator=SVC(kernel="linear", C=kw.get("C",1.0), class_weight="balanced", probability=False),
        method="sigmoid", cv=3
    ), [
        {"C":0.5}, {"C":1.0}, {"C":2.0}
    ]),
    ("LogisticRegression", LogisticRegression, [
        {"C":0.1, "solver":"liblinear", "penalty":"l2", "class_weight":"balanced", "max_iter":4000, "random_state":RNG_SEED},
        {"C":0.5, "solver":"liblinear", "penalty":"l2", "class_weight":"balanced", "max_iter":4000, "random_state":RNG_SEED},
        {"C":1.0, "solver":"liblinear", "penalty":"l2", "class_weight":"balanced", "max_iter":4000, "random_state":RNG_SEED},
    ]),
    ("RidgeClassifier", RidgeClassifier, [
        {"alpha":1.0}, {"alpha":5.0}, {"alpha":10.0},
    ]),
    ("SGDClassifier", SGDClassifier, [
        {"loss":"hinge",    "alpha":1e-4, "class_weight":"balanced", "max_iter":4000, "tol":1e-4, "random_state":RNG_SEED},
        {"loss":"hinge",    "alpha":1e-5, "class_weight":"balanced", "max_iter":4000, "tol":1e-4, "random_state":RNG_SEED},
        {"loss":"log_loss", "alpha":1e-5, "class_weight":"balanced", "max_iter":4000, "tol":1e-4, "random_state":RNG_SEED},
    ]),
    ("PassiveAggressive", PassiveAggressiveClassifier, [
        {"C":0.5, "class_weight":"balanced", "max_iter":4000, "tol":1e-4, "random_state":RNG_SEED},
        {"C":1.0, "class_weight":"balanced", "max_iter":4000, "tol":1e-4, "random_state":RNG_SEED},
        {"C":2.0, "class_weight":"balanced", "max_iter":4000, "tol":1e-4, "random_state":RNG_SEED},
    ]),
    ("LDA", LDA, [
        {"solver":"lsqr", "shrinkage":"auto"},
    ]),
    ("QDA", QDA, [
        {"reg_param":0.05}, {"reg_param":0.10},
    ]),
    ("GaussianNB", GaussianNB, [
        {"var_smoothing":1e-9}, {"var_smoothing":1e-8}
    ]),
    ("NearestCentroid", NearestCentroid, [{"metric":"euclidean"}]),
    ("KNeighborsClassifier", KNeighborsClassifier, [
        {"n_neighbors":11, "metric":"cosine", "weights":"distance", "algorithm":"brute"},
        {"n_neighbors":21, "metric":"cosine", "weights":"distance", "algorithm":"brute"},
    ]),
]

# ------------------------------
# Tuning loop
# ------------------------------
def tune_ovr_model(label: str, base, param_grid: List[Dict[str, Any]], min_accept: float = 0.01):
    best = None
    for params in param_grid:
        try:
            S_core_raw, S_eval_raw, clf, used_oof = _get_scores(label, base, params)
        except Exception as e:
            print(f"[skip] {label} {params} -> score error: {e}")
            continue

        # Calibrate raw scores to [0,1] using (OOF if available) core scores
        cdfs = _fit_cdf_per_class(S_core_raw)
        S_core01 = _apply_cdf_per_class(S_core_raw, cdfs)
        S_eval01 = _apply_cdf_per_class(S_eval_raw, cdfs)

        cands = []
        for p in P_GRID:
            thr = thresholds_from_percentile_weighted(S_core01, Y_core, p, ROW_W if USE_ROW_WEIGHTS else np.ones_like(ROW_W))
            Yb_c, Yb_e, out_c, out_e, scs_c, scs_e, parts_c, parts_e = evaluate_with_thresholds(
                S_core01, S_eval01, thr, core_words, eval_verbs
            )
            if out_e["acc"]["acceptance"] < min_accept:
                continue

            obj_e = _objective(scs_e - SCS_EMPTY_LEVEL_PEN * parts_e["empty_levels"], out_e, TARGETS)

            cands.append({
                "p": p, "thr": thr, "scs_c": scs_c, "scs_e": scs_e, "obj_e": obj_e,
                "out_c": out_c, "out_e": out_e, "parts_c": parts_c, "parts_e": parts_e,
                "Yb_c": Yb_c, "Yb_e": Yb_e, "clf": clf,
                "S_core": S_core01, "S_eval": S_eval01, "params": params,
                "cdfs": cdfs, "row_w": ROW_W if USE_ROW_WEIGHTS else np.ones_like(ROW_W),
                "Model": label, "Mode": f"P{p}",   # <-- ensure presence for table
            })

        if not cands:
            continue

        # keep near-top by SCS, then sort by acceptance and penalized objective
        cands.sort(key=lambda d: d["scs_e"], reverse=True)
        top = cands[0]["scs_e"]
        near = [d for d in cands if d["scs_e"] >= top - SCS_TIE_EPS]
        near.sort(key=lambda d: (d["obj_e"], d["out_e"]["acc"]["acceptance"]), reverse=True)
        sel = near[0]

        if (best is None) or (sel["obj_e"] > best["obj_e"]) or (
            abs(sel["obj_e"] - best["obj_e"]) < 1e-12 and sel["scs_e"] > best["scs_e"]
        ):
            best = sel
    return best

# sanity: norms for cosine pipelines
def _mean_norm(A): 
    n = np.linalg.norm(A, axis=1) + 1e-12
    return float(np.mean(n))

if not (0.95 <= _mean_norm(X_core) <= 1.05 and 0.95 <= _mean_norm(X_eval) <= 1.05):
    print("[warn] Embeddings seem not L2-normalized; cosine-based pieces may degrade.")

records: List[Dict[str, Any]] = []

for label, base, grid in MODEL_GRID:
    print(f"=== Tuning {label} ===")
    rec = tune_ovr_model(label, base, grid, min_accept=0.01)
    if rec is None:
        print(f"[warn] No valid configuration for {label}")
        continue
    records.append(rec)

print("=== Tuning Legacy kNN (unweighted) ===")
rec_knn = tune_legacy_knn(use_freq=False, min_accept=0.01)
if rec_knn is not None:
    records.append(rec_knn)

if 'core_freq' in globals():
    print("=== Tuning Legacy kNN (freq-weighted) ===")
    rec_knn_w = tune_legacy_knn(use_freq=True, min_accept=0.01)
    if rec_knn_w is not None:
        # prefer weighted if better by obj/SCS/acceptance
        def _key(r): return (r["obj_e"], r["scs_e"], r["out_e"]["acc"]["acceptance"])
        if rec_knn is None or _key(rec_knn_w) > _key(rec_knn):
            if rec_knn is None: records.append(rec_knn_w)
            else: records[-1] = rec_knn_w

# Validate/clean records to avoid KeyError: 'Model'
records = [r for r in records if isinstance(r, dict) and "Model" in r and "Mode" in r]

if not records:
    raise RuntimeError("No valid model configuration produced a usable candidate. Consider widening TARGET bands or P_GRID.")

def _row(rec):
    oc, oe = rec["out_c"], rec["out_e"]
    pc, pe = rec["parts_c"], rec["parts_e"]
    return dict(
        Model=f"{rec['Model']} ({rec['Mode']})",
        Params=str(rec["params"] if "params" in rec else rec.get("Params","{}")),
        Core_SCS=rec["scs_c"],  Core_Acceptance=oc["acc"]["acceptance"],  Core_EntryRate=oc["acc"]["entry_rate"], Core_en_over_ac=oc["acc"]["en_over_ac"],
        Core_D=pc["D"], Core_M=pc["M"], Core_trend=pc["trend"], Core_EmptyLevels=pc["empty_levels"],
        Eval_SCS=rec["scs_e"], Eval_Acceptance=oe["acc"]["acceptance"], Eval_EntryRate=oe["acc"]["entry_rate"], Eval_en_over_ac=oe["acc"]["en_over_ac"],
        Eval_D=pe["D"], Eval_M=pe["M"], Eval_trend=pe["trend"], Eval_EmptyLevels=pe["empty_levels"],
        PenalizedObj=rec.get("obj_e", _objective(rec["scs_e"], rec["out_e"], TARGETS))
    )

df_models = pd.DataFrame([_row(r) for r in records])
try:
    display(df_models.sort_values(["PenalizedObj","Eval_SCS","Eval_Acceptance"], ascending=[False, False, False]))
except Exception:
    print(df_models.sort_values(["PenalizedObj","Eval_SCS","Eval_Acceptance"], ascending=[False, False, False]).to_string(index=False))

df_pareto = _pareto_front(df_models, cols=[("Eval_SCS", False), ("Eval_Acceptance", False)])
print("\nPareto-front (Eval_SCS vs Acceptance):")
try:
    display(df_pareto.sort_values(["Eval_SCS","Eval_Acceptance"], ascending=[False, False]))
except Exception:
    print(df_pareto.sort_values(["Eval_SCS","Eval_Acceptance"], ascending=[False, False]).to_string(index=False))

# Final pick: maximize PenalizedObj; tie -> higher Eval_SCS then Acceptance
df_pick = df_models.sort_values(["PenalizedObj","Eval_SCS","Eval_Acceptance"], ascending=[False, False, False])
winner_row = df_pick.iloc[0]
print("\n>> Selected winner (taxonomy-aware objective):")
try:
    display(winner_row.to_frame().T)
except Exception:
    print(winner_row.to_frame().T)

# ------------------------------
# Expose production artifacts
# ------------------------------
winner_name = winner_row["Model"]
_w = next(r for r in records if f"{r['Model']} ({r['Mode']})" == winner_name)

winner_model       = _w.get("clf", None)          # None for Legacy kNN (vote)
winner_params      = _w.get("params", _w.get("Params", {}))
winner_thresholds  = _w.get("thr", None)          # thresholds in [0,1] for OvR models; None for kNN
Ybin_winner_core   = _w["Yb_c"]
Ybin_winner_eval   = _w["Yb_e"]
S_core_winner      = _w.get("S_core", None)       # calibrated scores (OvR) or None (kNN)
S_eval_winner      = _w.get("S_eval", None)       # calibrated scores (OvR) or None (kNN)
winner_cdfs        = _w.get("cdfs", None)         # apply to raw scores before thresholds (OvR)
winner_row_weights = _w.get("row_w", ROW_W)

print(f"\nArtifacts ready for production:\n  {winner_name} | params={winner_params} | thr={winner_thresholds}")
if winner_thresholds is not None:
    print("Production rule (OvR): score raw -> apply winner_cdfs -> compare to winner_thresholds -> Ybin")
else:
    print("Production rule (Legacy kNN): run with tuned (K,thresh,cutoff) -> Ybin")


Data-driven TARGETS: {'ACC_BAND': (0.92, 0.99), 'ENAC_BAND': (1.4800558659217877, 1.6400558659217879), 'SCS_TIE_EPS': 0.02, 'PEN_ACC': 5.0, 'PEN_ENAC': 2.5}
Adjusted TARGETS: {'ACC_BAND': (0.85, 0.851), 'ENAC_BAND': (1.4800558659217877, 1.6400558659217879), 'SCS_TIE_EPS': 0.02, 'PEN_ACC': 5.0, 'PEN_ENAC': 2.5}
[weights] use=True | min=0.721 mean=1.000 max=1.278
effective rank X_core: 358
=== Tuning LinearSVC ===
=== Tuning CalibratedSVC ===
=== Tuning LogisticRegression ===
=== Tuning RidgeClassifier ===
=== Tuning SGDClassifier ===
=== Tuning PassiveAggressive ===
=== Tuning LDA ===
=== Tuning QDA ===
=== Tuning GaussianNB ===
=== Tuning NearestCentroid ===
=== Tuning KNeighborsClassifier ===
=== Tuning Legacy kNN (unweighted) ===
=== Tuning Legacy kNN (freq-weighted) ===


,Model,Params,Core_SCS,Core_Acceptance,Core_EntryRate,Core_en_over_ac,Core_D,Core_M,Core_trend,Core_EmptyLevels,Eval_SCS,Eval_Acceptance,Eval_EntryRate,Eval_en_over_ac,Eval_D,Eval_M,Eval_trend,Eval_EmptyLevels,PenalizedObj
4,SGDClassifier (P60),"{'loss': 'hinge', 'alpha': 1e-05, 'class_weigh...",0.969034,0.826816,1.360335,1.645270,0.217238,0.122841,0.95,0,1.036728,0.825658,1.307018,1.583001,0.227672,0.105294,1.00,0,1.033765
5,PassiveAggressive (P60),"{'C': 1.0, 'class_weight': 'balanced', 'max_it...",0.992507,0.832402,1.379888,1.657718,0.210806,0.132814,1.00,0,0.982355,0.764254,1.141996,1.494261,0.212608,0.104328,0.95,0,0.945594
0,LinearSVC (P50),"{'C': 2.0, 'class_weight': 'balanced', 'max_it...",0.902677,0.891061,1.673184,1.877743,0.194491,0.167103,0.95,0,0.945315,0.808662,1.270833,1.571525,0.182111,0.110780,0.95,0,0.936771
10,KNeighborsClassifier (P55),"{'n_neighbors': 11, 'metric': 'cosine', 'weigh...",1.017657,0.896648,1.418994,1.582555,0.242508,0.101602,0.95,0,0.955735,0.786184,1.098136,1.396792,0.174152,0.092395,0.95,0,0.918041
2,LogisticRegression (P45),"{'C': 0.1, 'solver': 'liblinear', 'penalty': '...",0.996074,0.891061,1.709497,1.918495,0.243043,0.126237,0.95,0,1.009138,0.709978,1.063048,1.497297,0.234761,0.062286,0.90,0,0.911107
3,RidgeClassifier (P60),{'alpha': 1.0},1.093085,0.748603,1.094972,1.462687,0.299787,0.084435,0.95,0,1.016426,0.510965,0.629386,1.231760,0.260072,0.039366,0.85,0,0.287574
1,CalibratedSVC (P60),{'C': 2.0},1.040354,0.743017,1.145251,1.541353,0.258983,0.095371,0.95,0,1.026818,0.483004,0.605263,1.253121,0.279903,0.048950,0.85,0,0.224641
8,GaussianNB (P55),{'var_smoothing': 1e-09},1.043073,0.770950,1.268156,1.644928,0.280902,0.076672,0.90,0,1.072039,0.452851,0.590461,1.303874,0.277745,0.042683,0.90,0,0.205802
6,LDA (P60),"{'solver': 'lsqr', 'shrinkage': 'auto'}",0.452124,0.756983,1.659218,2.191882,0.209397,0.118651,0.30,0,0.498629,0.674342,1.271930,1.886179,0.228717,0.089704,0.30,0,0.192909
7,QDA (P60),{'reg_param': 0.05},0.449760,0.782123,1.648045,2.107143,0.209528,0.120617,0.30,0,0.503323,0.672697,1.290022,1.917685,0.228778,0.084893,0.30,0,0.153447



Pareto-front (Eval_SCS vs Acceptance):


,Model,Params,Core_SCS,Core_Acceptance,Core_EntryRate,Core_en_over_ac,Core_D,Core_M,Core_trend,Core_EmptyLevels,Eval_SCS,Eval_Acceptance,Eval_EntryRate,Eval_en_over_ac,Eval_D,Eval_M,Eval_trend,Eval_EmptyLevels,PenalizedObj
11,Legacy kNN (K=5/th=0.32/c=32),"{'K': 5, 'thresh': 0.32, 'cutoff': 32, 'use_fr...",1.203721,0.262570,0.365922,1.393617,0.468088,0.095533,0.95,1,1.182699,0.064693,0.079496,1.228814,0.379553,0.067637,1.0,1,-2.058643
8,GaussianNB (P55),{'var_smoothing': 1e-09},1.043073,0.770950,1.268156,1.644928,0.280902,0.076672,0.90,0,1.072039,0.452851,0.590461,1.303874,0.277745,0.042683,0.9,0,0.205802
4,SGDClassifier (P60),"{'loss': 'hinge', 'alpha': 1e-05, 'class_weigh...",0.969034,0.826816,1.360335,1.645270,0.217238,0.122841,0.95,0,1.036728,0.825658,1.307018,1.583001,0.227672,0.105294,1.0,0,1.033765
9,NearestCentroid (P45),{'metric': 'euclidean'},0.697803,1.000000,3.329609,3.329609,0.167325,0.308696,0.90,0,0.779631,1.000000,2.927083,2.927083,0.159435,0.216339,0.9,0,-3.472473



>> Selected winner (taxonomy-aware objective):


,Model,Params,Core_SCS,Core_Acceptance,Core_EntryRate,Core_en_over_ac,Core_D,Core_M,Core_trend,Core_EmptyLevels,Eval_SCS,Eval_Acceptance,Eval_EntryRate,Eval_en_over_ac,Eval_D,Eval_M,Eval_trend,Eval_EmptyLevels,PenalizedObj
4,SGDClassifier (P60),"{'loss': 'hinge', 'alpha': 1e-05, 'class_weigh...",0.969034,0.826816,1.360335,1.64527,0.217238,0.122841,0.95,0,1.036728,0.825658,1.307018,1.583001,0.227672,0.105294,1.0,0,1.033765



Artifacts ready for production:
  SGDClassifier (P60) | params={'loss': 'hinge', 'alpha': 1e-05, 'class_weight': 'balanced', 'max_iter': 4000, 'tol': 0.0001, 'random_state': 13} | thr=[0.91086351 0.80501393 0.63788301 0.72980501 0.67688022 0.88300836]
Production rule (OvR): score raw -> apply winner_cdfs -> compare to winner_thresholds -> Ybin


In [42]:
# === SAVE WINNER BUNDLE (model + calibration + thresholds + metadata) ===
import os, json, time, numpy as np, joblib, pickle, inspect, textwrap
from pathlib import Path

# ---------- sanity checks ----------
reqs = ["winner_model", "winner_cdfs", "winner_thresholds", "winner_params"]
for r in reqs:
    if r not in globals():
        raise RuntimeError(f"Missing '{r}'. Run tuning/parity first.")
if winner_model is None:
    raise RuntimeError("winner_model is None (Legacy kNN cannot be serialized as a scoring model in this path).")

# Optional artifacts (saved if present)
thr_oof = globals().get("winner_thresholds_oof", None)
S_core  = globals().get("S_core_winner", None)
S_eval  = globals().get("S_eval_winner", None)
row_w   = globals().get("winner_row_weights", None)

# ---------- levels & names ----------
LEVELS = ["Kn","Cm","Ap","An","Sn","Ev"]
LEVEL_NAMES = {"Kn":"knowledge","Cm":"comprehension","Ap":"application","An":"analysis","Sn":"synthesis","Ev":"evaluation"}

# ---------- paths ----------
base_dir = Path("results") / "BT4_structural_best" / "artifacts"
base_dir.mkdir(parents=True, exist_ok=True)
ts = time.strftime("%Y-%m-%d_%H-%M-%S")
bundle_name = f"{winner_name.replace(' ','_').replace('(','').replace(')','')}_{ts}"
bundle_dir = base_dir / bundle_name
bundle_dir.mkdir(parents=True, exist_ok=True)

# ---------- save model ----------
model_path = bundle_dir / "model.pkl"
joblib.dump(winner_model, model_path)

# ---------- save calibration CDFs ----------
# winner_cdfs is List[(xs: np.ndarray, cdf: np.ndarray)] for each class.
# Save as a JSON-safe structure: list of dicts with float lists.
cdfs_serializable = []
for (xs, cdf) in winner_cdfs if winner_cdfs is not None else []:
    xs = np.asarray(xs, dtype=float)
    cdf = np.asarray(cdf, dtype=float)
    cdfs_serializable.append({
        "xs": xs.tolist(),
        "cdf": cdf.tolist()
    })
cdfs_path = bundle_dir / "calibration_cdfs.json"
with open(cdfs_path, "w", encoding="utf-8") as f:
    json.dump(cdfs_serializable, f, ensure_ascii=False)

# ---------- save thresholds (tuning + oof if available) ----------
thr_obj = {
    "levels": LEVELS,
    "thresholds_tuning": np.asarray(winner_thresholds, dtype=float).tolist(),
}
if thr_oof is not None:
    thr_obj["thresholds_oof_aligned"] = np.asarray(thr_oof, dtype=float).tolist()
thr_path = bundle_dir / "thresholds.json"
with open(thr_path, "w", encoding="utf-8") as f:
    json.dump(thr_obj, f, ensure_ascii=False, indent=2)

# ---------- save row weights (for reproducibility of weighted percentiles) ----------
if row_w is not None:
    np.save(bundle_dir / "row_weights.npy", np.asarray(row_w, dtype=float))

# ---------- save score snapshots (optional, for audit) ----------
if S_core is not None:
    np.save(bundle_dir / "S_core_calibrated.npy", np.asarray(S_core, dtype=float))
if S_eval is not None:
    np.save(bundle_dir / "S_eval_calibrated.npy", np.asarray(S_eval, dtype=float))

# ---------- metadata ----------
def _mean_norm(A):
    if A is None: return None
    n = np.linalg.norm(A, axis=1) + 1e-12
    return float(np.mean(n))

meta = {
    "created_at": ts,
    "winner_name": winner_name,
    "winner_params": winner_params,
    "levels": LEVELS,
    "level_names": LEVEL_NAMES,
    "use_row_weights": row_w is not None,
    "has_oof_thresholds": thr_oof is not None,
    "calibration": {
        "type": "rank_cdf_per_class",
        "note": "Apply CDFs to raw decision scores to map to [0,1] before thresholding.",
        "cdf_count": len(cdfs_serializable) if winner_cdfs is not None else 0
    },
    "thresholds": {
        "tuning_source": str(thr_path),
        "oof_present": thr_oof is not None
    },
    "artifact_paths": {
        "model": str(model_path),
        "cdfs": str(cdfs_path),
        "thresholds": str(thr_path),
        "row_weights": str(bundle_dir / "row_weights.npy") if row_w is not None else None,
        "S_core_calibrated": str(bundle_dir / "S_core_calibrated.npy") if S_core is not None else None,
        "S_eval_calibrated": str(bundle_dir / "S_eval_calibrated.npy") if S_eval is not None else None,
    }
}
with open(bundle_dir / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# ---------- write a tiny scorer helper for production ----------
scorer_code = textwrap.dedent("""
    import json, numpy as np, joblib
    from pathlib import Path

    LEVELS = ["Kn","Cm","Ap","An","Sn","Ev"]

    def _load_cdfs(path):
        with open(path, "r", encoding="utf-8") as f:
            raw = json.load(f)
        cdfs = []
        for d in raw:
            xs = np.asarray(d["xs"], dtype=float)
            cdf = np.asarray(d["cdf"], dtype=float)
            cdfs.append((xs, cdf))
        return cdfs

    def apply_cdfs(S, cdfs):
        S = np.nan_to_num(np.asarray(S, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
        S01 = np.zeros_like(S, dtype=float)
        for j in range(S.shape[1]):
            xs, cdf = cdfs[j]
            S01[:, j] = np.interp(S[:, j], xs, cdf, left=cdf[0], right=cdf[-1])
        return np.clip(S01, 0.0, 1.0)

    class BT4Scorer:
        def __init__(self, bundle_dir, use_oof_thresholds=False):
            bundle = Path(bundle_dir)
            self.model = joblib.load(bundle / "model.pkl")
            self.cdfs  = _load_cdfs(bundle / "calibration_cdfs.json")
            with open(bundle / "thresholds.json", "r", encoding="utf-8") as f:
                thr = json.load(f)
            if use_oof_thresholds and "thresholds_oof_aligned" in thr:
                self.thresholds = np.asarray(thr["thresholds_oof_aligned"], dtype=float)
                self.thr_source = "OOF_ALIGNED"
            else:
                self.thresholds = np.asarray(thr["thresholds_tuning"], dtype=float)
                self.thr_source = "TUNING"

        def score_raw(self, X):
            # Uses decision_function if available; else predict_proba; else predict.
            m = self.model
            if hasattr(m, "decision_function"):
                S = m.decision_function(X)
            elif hasattr(m, "predict_proba"):
                P = m.predict_proba(X)
                # OneVsRest: list of (n,2) arrays → take prob of class 1
                if isinstance(P, list):
                    S = np.column_stack([p[:,1] if p.ndim == 2 and p.shape[1] >= 2 else p.ravel() for p in P])
                else:
                    S = P
            else:
                S = m.predict(X).astype(float)
            S = np.asarray(S, dtype=float)
            if S.ndim == 1: S = S.reshape(-1, len(LEVELS))
            return S

        def score_calibrated(self, X):
            S = self.score_raw(X)
            return apply_cdfs(S, self.cdfs)

        def predict_binary(self, X):
            S01 = self.score_calibrated(X)
            Yb = (S01 >= self.thresholds.reshape(1, -1)).astype(int)
            return Yb, S01

        def suggest_binary(self, X, alpha=0.85):
            S01 = self.score_calibrated(X)
            thr_soft = self.thresholds * float(alpha)
            Yb = (S01 >= thr_soft.reshape(1, -1)).astype(int)
            return Yb, S01
""").strip("\n")

scorer_path = bundle_dir / "bt4_scorer.py"
with open(scorer_path, "w", encoding="utf-8") as f:
    f.write(scorer_code)

# ---------- report ----------
print("Winner bundle saved.")
print("Bundle directory:", str(bundle_dir))
print("  - model.pkl")
print("  - calibration_cdfs.json")
print("  - thresholds.json", "(+ OOF in same file)" if thr_oof is not None else "")
if row_w is not None: print("  - row_weights.npy")
if S_core is not None: print("  - S_core_calibrated.npy")
if S_eval is not None: print("  - S_eval_calibrated.npy")
print("  - metadata.json")
print("  - bt4_scorer.py")

# Optional: write a 'latest' symlink/marker for convenience (best effort)
try:
    latest_link = base_dir / "LATEST"
    if latest_link.exists() or latest_link.is_symlink():
        latest_link.unlink()
    latest_link.symlink_to(bundle_dir.name)  # relative link
    print("Updated symlink:", str(latest_link), "->", bundle_dir.name)
except Exception as _e:
    pass


Winner bundle saved.
Bundle directory: results\BT4_structural_best\artifacts\SGDClassifier_P60_2025-10-12_01-47-29
  - model.pkl
  - calibration_cdfs.json
  - thresholds.json (+ OOF in same file)
  - row_weights.npy
  - S_core_calibrated.npy
  - S_eval_calibrated.npy
  - metadata.json
  - bt4_scorer.py
Updated symlink: results\BT4_structural_best\artifacts\LATEST -> SGDClassifier_P60_2025-10-12_01-47-29


In [ ]:
# === PRODUCTION (prefers parity/tuning by default; can switch to OOF+Aligned) — with Suggested Domains ===
import numpy as np, pandas as pd, os, time

# --------------------------
# CONFIG TOGGLES
# --------------------------
PREFER_PARITY_THRESH = True      # True → tuning thresholds; False → use OOF-aligned if available
PREFER_PARITY_SCORES = True      # True → use tuning-calibrated scores if present
APPLY_TOP2_MARGIN    = True      # Post-process: collapse to single label if (top1 - top2) >= margin
TOP2_MARGIN_DEFAULT  = 0.12
SUGGEST_ALPHA        = 0.85      # "Soft suggestion" threshold = alpha * hard threshold (per class)

# --------------------------
# constants & helpers
# --------------------------
LEVELS = ["Kn","Cm","Ap","An","Sn","Ev"]
LEVEL_NAMES = {"Kn":"knowledge","Cm":"comprehension","Ap":"application","An":"analysis","Sn":"synthesis","Ev":"evaluation"}

def _accepted_pattern(bits: np.ndarray) -> str:
    return "".join(str(int(b)) for b in bits.tolist())

def _choose_primary(scores01: np.ndarray, mask_bin: np.ndarray) -> list:
    """Primary = argmax of scores among 1s in mask; '' if none."""
    prim = []
    for i in range(scores01.shape[0]):
        if mask_bin[i].sum() == 0:
            prim.append("")
        else:
            idx = np.argmax(scores01[i] * mask_bin[i])
            prim.append(LEVELS[idx])
    return prim

def _other_domains(mask_bin: np.ndarray, primary: list) -> list:
    out = []
    prim_idx = {i: (LEVELS.index(p) if p in LEVELS else -1) for i,p in enumerate(primary)}
    for i in range(mask_bin.shape[0]):
        if mask_bin[i].sum() <= 1:
            out.append("")
            continue
        others = [LEVELS[j] for j in range(6) if mask_bin[i, j] == 1 and j != prim_idx[i]]
        out.append(", ".join(others))
    return out

def _ensure_eval_verb_col(df_eval):
    for c in ["verb","Verb","word","token","WORD","VERB"]:
        if c in df_eval.columns: return c
    raise RuntimeError("No verb column found in df_eval.")

def _apply_top2_margin(S01, Yb, delta=0.12):
    """If >=2 labels accepted and (top1 - top2) >= delta, collapse to single label."""
    Yb2 = Yb.copy()
    for i in range(S01.shape[0]):
        idx = np.where(Yb2[i] == 1)[0]
        if idx.size >= 2:
            s = S01[i, idx]
            o = np.argsort(s)[::-1]
            top, second = idx[o[0]], idx[o[1]]
            if (S01[i, top] - S01[i, second]) >= delta:
                Yb2[i, :] = 0
                Yb2[i, top] = 1
    return Yb2

# --------------------------
# pull artifacts 
# --------------------------
for req in ["winner_name", "winner_params"]:
    if req not in globals():
        raise RuntimeError(f"Missing artifact '{req}'. Run the tuning/parity cell first.")

vcol_eval = _ensure_eval_verb_col(df_eval)
verbs = [str(v).strip().lower() for v in df_eval[vcol_eval].tolist()]

# Detect OOF artifacts
HAS_OOF = ("winner_thresholds_oof" in globals() and (winner_thresholds_oof is not None))
USE_OOF = (not PREFER_PARITY_THRESH) and HAS_OOF

# --------------------------
# choose scores + thresholds
# --------------------------
# Scores
if PREFER_PARITY_SCORES and ("S_eval_winner_tuning" in globals()) and (S_eval_winner_tuning is not None):
    S01 = np.asarray(S_eval_winner_tuning, dtype=float)   # tuning-calibrated scores (parity path)
    score_src = "TUNING"
elif ("S_eval_winner" in globals()) and (S_eval_winner is not None):
    S01 = np.asarray(S_eval_winner, dtype=float)          # last available artifact
    score_src = "CURRENT"
else:
    raise RuntimeError("Missing S_eval_winner (and S_eval_winner_tuning). Re-run tuning/parity or OOF cell.")

# Thresholds
if USE_OOF:
    thr_vec = np.asarray(winner_thresholds_oof, dtype=float)
    chosen = f"{winner_name} | OOF-calibrated + aligned thr"
    print("[prod] Using OOF+Aligned thresholds")
else:
    if 'winner_thresholds' not in globals() or winner_thresholds is None:
        raise RuntimeError("Missing winner_thresholds for parity/tuning path.")
    thr_vec = np.asarray(winner_thresholds, dtype=float)
    chosen = f"{winner_name} | calibrated thresholds (tuning)"
    print("[prod] Using parity/tuning thresholds")

# --------------------------
# decisions (HARD) + optional postproc
# --------------------------
Ybin_hard = (S01 >= thr_vec.reshape(1, -1)).astype(int)

if APPLY_TOP2_MARGIN:
    delta = None
    if "POSTPROC" in globals() and isinstance(POSTPROC, dict):
        delta = float(POSTPROC.get("top2_margin", TOP2_MARGIN_DEFAULT))
    if delta is None: delta = TOP2_MARGIN_DEFAULT
    Ybin_hard = _apply_top2_margin(S01, Ybin_hard, delta=delta)
    chosen += f" + top2-margin({delta:.3f})"

# --------------------------
# decisions (SOFT / SUGGESTED) — independent of top2 collapse
# --------------------------
soft_thr = thr_vec * float(SUGGEST_ALPHA)
Ybin_soft = (S01 >= soft_thr.reshape(1, -1)).astype(int)

# --------------------------
# build export table
# --------------------------
S01r = np.round(S01, 3)
export = pd.DataFrame({"Verb": verbs})

# scores + hard accept columns
for j, lv in enumerate(LEVELS):
    export[f"score_{lv}"]      = S01r[:, j]
    export[f"accept_{lv}"]     = Ybin_hard[:, j].astype(int)
    export[f"cal_score_{lv}"]  = np.round(S01[:, j], 6)
    export[f"thr_{lv}"]        = np.round(thr_vec[j], 6)
    export[f"meets_thr_{lv}"]  = (S01[:, j] >= thr_vec[j]).astype(int)

# hard summary columns
export["AcceptedCount"]   = Ybin_hard.sum(axis=1).astype(int)
export["Accepted"]        = export["AcceptedCount"] > 0
export["AcceptedPattern"] = [_accepted_pattern(Ybin_hard[i]) for i in range(Ybin_hard.shape[0])]
hard_primary              = _choose_primary(S01, Ybin_hard)
export["PrimaryDomain"]   = hard_primary
export["OtherDomains"]    = _other_domains(Ybin_hard, hard_primary)

# SOFT/SUGGESTED columns
export["SoftAcceptedCount"] = Ybin_soft.sum(axis=1).astype(int)
export["SoftAccepted"]      = export["SoftAcceptedCount"] > 0
export["SoftPattern"]       = [_accepted_pattern(Ybin_soft[i]) for i in range(Ybin_soft.shape[0])]
soft_primary                = _choose_primary(S01, Ybin_soft)
export["SoftPrimaryDomain"] = soft_primary
export["SoftOtherDomains"]  = _other_domains(Ybin_soft, soft_primary)

# keep original labels (if present) for traceability
trace_cols = [c for c in df_eval.columns if c in LEVELS]
if trace_cols:
    export = export.merge(
        df_eval[[vcol_eval] + trace_cols].rename(columns={vcol_eval: "Verb"}),
        on="Verb", how="left"
    )

# --------------------------
# suggestion-only summary (soft > alpha * hard threshold, but hard not accepted)
# --------------------------
suggested_only = (export["AcceptedCount"] == 0) & export["SoftAccepted"]
suggested_only_count = int(suggested_only.sum())

# --------------------------
# summary (acceptance, distribution, per-domain)
# --------------------------
n = len(export)
accepted_words   = int(export["Accepted"].sum())
rejected_words   = int((~export["Accepted"]).sum())
fully_accepted   = int((export["AcceptedCount"] == 6).sum())
single_label     = int((export["AcceptedCount"] == 1).sum())
multi_label      = int((export["AcceptedCount"] >= 2).sum())
acc_rate         = float(export["Accepted"].mean()) if n else 0.0
entry_rate       = float(export["AcceptedCount"].mean()) if n else 0.0
hist             = export["AcceptedCount"].value_counts().reindex(range(0,7), fill_value=0).astype(int).to_dict()
per_domain_totals = {lv: int(export[f"accept_{lv}"].sum()) for lv in LEVELS}

print(f"Chosen setting: {chosen}")
print(f"[debug] score source: {score_src} | thr source: {'OOF' if USE_OOF else 'TUNING'} | thr min/max: {thr_vec.min():.4f}/{thr_vec.max():.4f}")
print(f"Acceptance rate: {acc_rate:.6f}   (accepted {accepted_words} / {n}, rejected {rejected_words})")
print(f"Entry rate:      {entry_rate:.6f}")
print(f"Fully accepted (all 6): {fully_accepted}")
print(f"Single-label items:     {single_label}")
print(f"Multi-label items:      {multi_label}")
print(f"Suggested-only (no hard accept but soft suggestions present @ {SUGGEST_ALPHA:.2f}×thr): {suggested_only_count}")
for k in range(0,7):
    print(f"# of words with {k} entries: {hist.get(k,0)}")
for lv in LEVELS:
    print(f"# of entries acquired by domain {LEVEL_NAMES[lv]}: {per_domain_totals[lv]}")

# --------------------------
# save
# --------------------------
agg_cols    = ["Verb","Accepted","AcceptedCount","AcceptedPattern","PrimaryDomain","OtherDomains",
               "SoftAccepted","SoftAcceptedCount","SoftPattern","SoftPrimaryDomain","SoftOtherDomains"]
score_cols  = [f"score_{lv}"  for lv in LEVELS]
debug_cols  = sum([[f"cal_score_{lv}", f"thr_{lv}", f"meets_thr_{lv}"] for lv in LEVELS], [])
accept_cols = [f"accept_{lv}" for lv in LEVELS]
orig_label_cols = [c for c in df_eval.columns if c in LEVELS]
label_cols_in_export = [c for c in orig_label_cols if c in export.columns]

cols_final = [c for c in (agg_cols + score_cols + debug_cols + accept_cols + label_cols_in_export) if c in export.columns]
export = export[cols_final]

out_dir = os.path.join("results","BT4_structural_best")
os.makedirs(out_dir, exist_ok=True)
ts = time.strftime("%Y-%m-%d_%H-%M-%S")
safe_winner = winner_name.replace(" ","_").replace("(","").replace(")","")
suffix = "OOF_ALIGNED" if USE_OOF else "TUNING"
out_path = os.path.join(out_dir, f"BT4_PROD_{suffix}_{safe_winner}_{ts}.csv")
export.to_csv(out_path, index=False)

print("\nSaved:", out_path)
display(export.head(25))


[prod] Using parity/tuning thresholds
Chosen setting: SGDClassifier (P60) | calibrated thresholds (tuning) + top2-margin(0.060)
[debug] score source: CURRENT | thr source: TUNING | thr min/max: 0.6379/0.9109
Acceptance rate: 0.825658   (accepted 1506 / 1824, rejected 318)
Entry rate:      1.019189
Fully accepted (all 6): 0
Single-label items:     1246
Multi-label items:      260
Suggested-only (no hard accept but soft suggestions present @ 0.85×thr): 240
# of words with 0 entries: 318
# of words with 1 entries: 1246
# of words with 2 entries: 185
# of words with 3 entries: 61
# of words with 4 entries: 10
# of words with 5 entries: 4
# of words with 6 entries: 0
# of entries acquired by domain knowledge: 49
# of entries acquired by domain comprehension: 373
# of entries acquired by domain application: 574
# of entries acquired by domain analysis: 259
# of entries acquired by domain synthesis: 485
# of entries acquired by domain evaluation: 119

Saved: results\BT4_structural_best\BT4_PR

,Verb,Accepted,AcceptedCount,AcceptedPattern,PrimaryDomain,OtherDomains,SoftAccepted,SoftAcceptedCount,SoftPattern,SoftPrimaryDomain,...,accept_Ap,accept_An,accept_Sn,accept_Ev,Kn,Cm,Ap,An,Sn,Ev
0,abandon,True,2,001001,Ev,Ap,True,3,011001,Ev,...,1,0,0,1,0,0,0,0,0,1
1,abbreviate,True,1,000010,Sn,,True,3,011010,Sn,...,0,0,1,0,0,1,0,0,1,0
2,abet,True,1,010000,Cm,,True,2,011000,Cm,...,0,0,0,0,0,0,0,0,0,0
3,abide,True,1,001000,Ap,,True,2,001001,Ap,...,1,0,0,0,0,0,0,0,0,0
4,ablate,True,1,010000,Cm,,True,1,010000,Cm,...,0,0,0,0,0,0,0,0,0,0
5,abort,True,1,000001,Ev,,True,3,001011,Ev,...,0,0,0,1,1,0,0,0,0,0
6,abound,False,0,000000,,,True,1,010000,Cm,...,0,0,0,0,0,0,0,0,0,0
7,absorb,False,0,000000,,,False,0,000000,,...,0,0,0,0,0,0,1,1,0,0
8,abut,True,1,010000,Cm,,True,3,011010,Cm,...,0,0,0,0,0,0,0,0,0,0
9,accelerate,True,1,000010,Sn,,True,2,001010,Sn,...,0,0,1,0,0,0,0,1,0,0
